# Axis 7 — Router-Guided Speculative Prefetching (Mac / MLX)

**Question (team page).** Can the router be used *ahead of time* to start loading the next layer's experts before the compute engine stalls on a miss?

**Hypothesis.** There is a usable window between "next experts can be guessed" and "next experts are needed". Prefetching in that window cuts the wait for correctly predicted experts to near zero, but mispredictions waste bandwidth (and cache slots) — a precision / recall trade-off set by how many experts we prefetch.

## Predictor: layer L+1's router on layer L's hidden state
At layer L, as soon as its input is known, apply **layer L+1's router** to it. Both layers normalise the residual stream with RMSNorm, which differs only by a per-channel weight, so the input layer L+1's router would see is reconstructed exactly up to the attention + MoE update in between:

$$ \hat{x}_{L+d} = \frac{x_L}{w_L} \odot w_{L+d}, \qquad \text{prediction} = \operatorname{top}_P\big(\mathrm{router}_{L+d}(\hat{x}_{L+d})\big) $$

Baseline predictor for comparison: *"the same experts layer L+1 used for the previous token"* (temporal reuse).

## Pipeline (real background I/O)
1. Layer L's routing is known → wait only for in-flight prefetches *this layer needs* → load layer L's remaining misses → launch layer L's expert compute (`mx.async_eval`).
2. Predict layer **L+D** (lookahead D = 1, 2 or 3 layers, using layer L+D's router the same way), admit those experts into the cache (they may evict others — that is the cost) and **read them on T background threads**.
3. When layer L+D runs, correctly predicted experts are hits. Slots still being written are protected from eviction until their read lands.

The knobs and what they trade:
* **P** (experts predicted per token): recall vs precision / wasted reads.
* **D** (lookahead): more time to hide each read vs a less accurate guess.
* **T** (reader threads): how much can be read in that time.

All modes use the same cache (LRU, K = 256) and produce identical text (prefetching changes *when* weights arrive, never which weights are used).

**Runtime.** ≈ 25–35 min full; `MOE_SMOKE=1` a few minutes.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlx.core as mx

# Find the repo and import the shared module (experiments/moe_cache.py).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments' / 'moe_cache.py').exists())
sys.path.insert(0, str(ROOT / 'experiments'))
import moe_cache as mc
importlib.reload(mc)

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-7-router-prefetching'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-7-router-prefetching') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(mc.system_info())

## Configuration

In [ ]:
K = 256
# name -> (P experts predicted per token, lookahead D layers, prefetch reader threads)
MODES = {'none': (0, 1, 1), 'P8-d1': (8, 1, 1), 'P8-d2': (8, 2, 1), 'P8-d3': (8, 3, 1),
         'P12-d2': (12, 2, 1), 'P8-d2-T4': (8, 2, 4)}
PMAX = 16
REQUESTS_PER_DOMAIN = 1          # x 4 kinds of request (max 2)
MAX_NEW_TOKENS = 48

if SMOKE:
    MODES, MAX_NEW_TOKENS = {'none': (0, 1, 1), 'P8-d2': (8, 2, 1)}, 8
print('K =', K, '| modes:', MODES)

## Model and correctness gate

In [ ]:
model, tokenizer, index = mc.load_model()
print(index.describe())
print(f'Non-expert weights resident: {mc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| MLX active memory: {mx.get_active_memory() / 1e9:.2f} GB')
print(f'Largest K that fits safely in this machine\'s RAM: {mc.max_safe_capacity(index)}')

# Correctness gate: one layer through the K-slot bank vs the stock mlx-lm layer
# (all 64 experts of that layer stacked). Must be bit-identical.
diff = mc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock mlx-lm: max |diff| = {diff}')

## Prefetcher and the prefetching MoE block

In [ ]:
from concurrent.futures import ThreadPoolExecutor

class Prefetcher:
    # Background reads into cache slots. A slot that is still being written is
    # "in flight" and is protected from eviction until its read lands.
    def __init__(self, cache, threads=1):
        self.cache, self.pool = cache, ThreadPoolExecutor(threads)
        self.inflight = {}                 # key -> future
        self.pending = set()               # prefetched, not used yet
        self.issued = self.useful = self.wasted = 0
        self.wait_ms = 0.0

    def _land(self, k):
        n, _ = self.inflight.pop(k).result()
        self.cache.bytes_loaded += n

    def issue(self, keys, protect):
        c = self.cache
        for k in [k for k, f in self.inflight.items() if f.done()]:
            self._land(k)
        guard = set(protect) | set(self.inflight)
        room = c.capacity - len(guard) - 64          # leave room for the next demand call
        keys = [k for k in dict.fromkeys(keys) if k not in c.slot_of][:max(0, room)]
        pinned = guard | set(keys)
        for k in keys:
            _, ev = c.policy.access(k, 1.0, pinned)
            if ev is not None:
                c.free.append(c.slot_of.pop(ev)); c.evictions += 1
                if ev in self.pending:
                    self.pending.discard(ev); self.wasted += 1
            slot = c.free.pop(); c.slot_of[k] = slot; self.pending.add(k)
            self.inflight[k] = self.pool.submit(c._load_one, k, slot)
        self.issued += len(keys)

    def wait_for(self, keys):
        t = time.perf_counter()
        for k in keys:
            if k in self.inflight:
                self._land(k)
        self.wait_ms += (time.perf_counter() - t) * 1000

    def finish(self):
        self.wait_for(list(self.inflight))
        self.wasted += len(self.pending); self.pending.clear()
        self.pool.shutdown()

NL = len(model.model.layers)
GATES = [layer.mlp.gate for layer in model.model.layers]
NORM_W = [layer.post_attention_layernorm.weight for layer in model.model.layers]
LOOKAHEADS = (1, 2, 3)

class PrefetchMoE(mc.CachedMoE):
    P, D, pf = 0, 1, None          # experts per token, lookahead (layers), prefetcher

    def __call__(self, x):
        B, L, Dm = x.shape
        x_flat = x.reshape(-1, Dm)
        inds, scores = self.route(x_flat)
        base = x_flat / NORM_W[self.layer]              # undo this layer's RMSNorm weight
        preds = {d: mx.argsort(-GATES[self.layer + d](base * NORM_W[self.layer + d]), axis=-1)[..., :PMAX]
                 for d in LOOKAHEADS if self.layer + d < NL}
        mx.eval(inds, scores, *preds.values())
        cache, pf = self.cache, self.pf
        ind_np, sc_np = np.array(inds), np.array(scores.astype(mx.float32))
        eids, inv = np.unique(ind_np, return_inverse=True)
        mass = np.bincount(inv.reshape(-1), weights=sc_np.reshape(-1), minlength=len(eids))
        keys = [(self.layer, int(e)) for e in eids]
        if pf is not None:
            pf.wait_for(keys)                            # only what this layer needs
            used = pf.pending & set(keys)
            pf.useful += len(used); pf.pending -= used
        t0 = time.perf_counter()                         # demand loads only (wait counted apart)
        h0, m0 = cache.hits, cache.misses
        slots = cache.ensure(keys, mass, protect=set(pf.inflight) if pf else ())
        if pf is not None:
            gone = {k for k in pf.pending if k not in cache.slot_of}
            pf.wasted += len(gone); pf.pending -= gone
        demand_ms = (time.perf_counter() - t0) * 1000
        lut = np.zeros(64, np.int32); lut[eids] = slots
        y = self.switch(x_flat, mx.array(lut[ind_np]), cache.G, cache.U, cache.D)
        y = (y * scores[..., None]).sum(axis=-2)
        mx.async_eval(y)
        rec = {'layer': self.layer, 'n': len(ind_np), 'phase': PHASE[0], 'step': STEP[0],
               'actual': eids.tolist(), 'hits': cache.hits - h0, 'misses': cache.misses - m0,
               'demand_ms': demand_ms}
        pn = {d: np.array(v) for d, v in preds.items()}
        rec['pred'] = {d: {p: np.unique(v[:, :p]).tolist() for p in (4, 8, 12, 16)} for d, v in pn.items()}
        if pf is not None and self.P and self.D in pn:
            pf.issue([(self.layer + self.D, int(e)) for e in np.unique(pn[self.D][:, :self.P])], set(keys))
        CALLS.append(rec)
        return y.reshape(B, L, Dm)

for i, layer in enumerate(model.model.layers):
    old = layer.mlp
    layer.mlp = PrefetchMoE(old.gate, i, None, old.top_k, old.norm_topk_prob)
CALLS, PHASE, STEP = [], ['prefill'], [0]
print('PrefetchMoE installed')

## Real runs, one per mode

In [ ]:
# A small mixed request set written for this axis (one or two per kind of request).
PROMPTS = {
    'math': ['A tank holds 1,200 litres and drains at 35 litres per minute. How long until it is empty? Show your steps.',
             'What is the sum of all whole numbers from 1 to 200? Explain the trick.'],
    'code': ['Write a Python function that groups a list of words into anagram sets.',
             'Write a Python function that validates a Sudoku board.'],
    'chat': ['What are the pros and cons of working from home?',
             'How do vaccines train the immune system?'],
    'writing': ['Write a short scene where two old friends meet at a train station after twenty years.',
                'Write a product description for a reusable water bottle.'],
}
reqs = mc.requests(PROMPTS, n_per_domain=REQUESTS_PER_DOMAIN, order='interleave')

def on_step(ph, i):
    PHASE[0], STEP[0] = ph, i

runs, all_calls, texts = [], {}, {}
for mname, (P, D, T) in MODES.items():
    mc.free_memory()
    cache = mc.ExpertCache(index, K, 'LRU')
    mc.attach_cache(model, cache, [])
    pf = Prefetcher(cache, T) if P else None
    PrefetchMoE.P, PrefetchMoE.D, PrefetchMoE.pf = P, D, pf
    CALLS.clear()
    out, dec_tok, dec_s, ttft = [], 0, 0.0, []
    for ri, r in enumerate(reqs):
        g = mc.generate(model, tokenizer, mc.chat_tokens(tokenizer, r['text']), MAX_NEW_TOKENS, on_step=on_step)
        out.append(g.tokens); dec_tok += len(g.step_ms); dec_s += g.decode_s; ttft.append(g.ttft_s)
        for c in CALLS[len(all_calls.get(mname, [])):]:
            c['req'] = ri
        all_calls[mname] = list(CALLS)
    if pf is not None:
        pf.finish()
    s = cache.summary()
    c = pd.DataFrame(CALLS)
    texts[mname] = out
    runs.append({'mode': mname, 'P': P, 'lookahead': D, 'threads': T, 'decode_tok_s': dec_tok / dec_s, 'ttft_mean_s': float(np.mean(ttft)),
                 'demand_misses': s['misses'], 'demand_hit_rate': s['hit_rate'],
                 'prefetched': pf.issued if pf else 0, 'useful': pf.useful if pf else 0,
                 'wasted': pf.wasted if pf else 0,
                 'wasted_GB': (pf.wasted if pf else 0) * index.expert_bytes / 1e9,
                 'prefetch_wait_s': (pf.wait_ms if pf else 0) / 1000,
                 'demand_load_s': c.demand_ms.sum() / 1000,
                 'stall_s': c.demand_ms.sum() / 1000 + (pf.wait_ms if pf else 0) / 1000})
    PrefetchMoE.pf = None
    cache.close(); mc.detach_cache(model)
    x = runs[-1]
    print(f'{mname:5s}: decode {x["decode_tok_s"]:.2f} tok/s | TTFT {x["ttft_mean_s"]:.2f}s | demand misses '
          f'{x["demand_misses"]} | prefetched {x["prefetched"]} (useful {x["useful"]}, wasted {x["wasted"]}) | '
          f'stall {x["stall_s"]:.1f}s')

runs = pd.DataFrame(runs)
runs.to_csv(RESULTS / 'axis7_runs.csv', index=False)
for m in MODES:
    assert texts[m] == texts['none'], f'{m}: prefetching changed the output!'
print('\nIdentical text in every mode (prefetching only changes when weights arrive).')

## Prediction quality (offline, from the no-prefetch run)
For every layer call, compare the experts predicted for the next layer with what that next layer actually used. Precision = useful share of what we would fetch; recall = share of the next layer's needs we would have covered. Baseline: the experts the same layer used for the previous token.

In [ ]:
base = pd.DataFrame(all_calls['none'])
rows = []
for i in range(len(base)):
    a = base.iloc[i]
    for d, by_p in a.pred.items():
        j = i + d
        if j >= len(base) or base.iloc[j].layer != a.layer + d:
            continue
        actual = set(base.iloc[j].actual)
        for P, pred in by_p.items():
            pred = set(pred)
            rows.append({'phase': a.phase, 'layer': a.layer + d, 'lookahead': d, 'P': P,
                         'predictor': f'router L+{d} on h_L',
                         'precision': len(pred & actual) / len(pred), 'recall': len(pred & actual) / len(actual)})
# temporal baseline: same layer, previous decode step of the same request
dec = base[base.phase == 'decode']
prev = {}
for _, r in dec.iterrows():
    k = (r.req, r.layer)
    if k in prev:
        p, a = set(prev[k]), set(r.actual)
        rows.append({'phase': 'decode', 'layer': r.layer, 'lookahead': 0, 'P': 8, 'predictor': 'previous token, same layer',
                     'precision': len(p & a) / len(p), 'recall': len(p & a) / len(a)})
    prev[k] = r.actual
acc = pd.DataFrame(rows)
acc.to_csv(RESULTS / 'axis7_prediction.csv', index=False)
summary = acc.groupby(['phase', 'predictor', 'P'])[['precision', 'recall']].mean()
display(summary.style.format('{:.1%}'))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for d in LOOKAHEADS:
    dd = acc[(acc.phase == 'decode') & (acc.lookahead == d)]
    dd = dd.groupby('P')[['precision', 'recall']].mean()
    axes[0].plot(dd.index, dd.recall, '-o', label=f'recall, lookahead {d}')
    axes[0].plot(dd.index, dd.precision, ':', alpha=0.7, label=f'precision, lookahead {d}')
tb = acc[acc.predictor == 'previous token, same layer'][['precision', 'recall']].mean()
axes[0].axhline(tb.recall, color='grey', ls='--', label=f'prev-token baseline ({tb.recall:.0%})')
axes[0].set_xlabel('experts predicted per token (P)'); axes[0].set_ylim(0, 1); axes[0].legend(fontsize=7)
axes[0].set_title('Decode: prediction quality vs lookahead')
per_layer = acc[(acc.phase == 'decode') & (acc.P == 8) & (acc.lookahead == 1)].groupby('layer').recall.mean()
axes[1].bar(per_layer.index, per_layer.values); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('predicted layer'); axes[1].set_ylabel('recall @ P=8'); axes[1].set_title('Recall by layer (lookahead 1)')
fig.tight_layout(); fig.savefig(RESULTS / 'axis7_prediction.png', dpi=150, bbox_inches='tight'); plt.show()

## Real cost and benefit

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
r = runs.set_index('mode')
axes[0].bar(r.index, r.decode_tok_s); axes[0].set_title('Decode throughput'); axes[0].set_ylabel('tokens / s')
axes[1].bar(r.index, r.demand_load_s, label='demand loads (critical path)')
axes[1].bar(r.index, r.prefetch_wait_s, bottom=r.demand_load_s, label='waiting for prefetch')
axes[1].set_title('Stall time'); axes[1].set_ylabel('s'); axes[1].legend(fontsize=8)
axes[2].bar(r.index, r.useful, label='useful prefetches'); axes[2].bar(r.index, r.wasted, bottom=r.useful, label='wasted')
axes[2].set_title('Prefetched experts'); axes[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS / 'axis7_cost.png', dpi=150, bbox_inches='tight'); plt.show()
runs.style.format(precision=2)

## Hypothesis check (descriptive)

In [ ]:
b = r.loc['none']
for m in r.index.drop('none'):
    x = r.loc[m]
    prec = x.useful / x.prefetched if x.prefetched else float('nan')
    print(f'{m}: stall {b.stall_s:.1f}s -> {x.stall_s:.1f}s ({x.stall_s / b.stall_s - 1:+.0%}) | '
          f'decode {b.decode_tok_s:.2f} -> {x.decode_tok_s:.2f} tok/s | TTFT {b.ttft_mean_s:.2f} -> {x.ttft_mean_s:.2f}s | '
          f'prefetch precision {prec:.0%} | wasted {x.wasted_GB:.1f} GB')
print('\nP trades recall for precision; lookahead trades accuracy for time to hide the read; '
      'reader threads raise how much can be read in that time.')